# PRÁCTICA 2

----
## EJERCICIO 1

In [1]:
import requests  # Importamos la librería requests, que nos permitirá hacer peticiones a la API.

# Nuestra clave API proporcionada por AEMET
api_key = ''

# 1. Obtener inventario de estaciones
# Definimos la URL del inventario de estaciones de AEMET para consultar todas las estaciones disponibles
url_inventario = 'https://opendata.aemet.es/opendata/api/valores/climatologicos/inventarioestaciones/todasestaciones'

# Creamos un diccionario de parámetros que incluye nuestra API Key para autenticar la petición
params = {'api_key': api_key}

# Ahora pedimos a la API el inventario de estaciones usando requests.get() con la URL y los parámetros
response = requests.get(url_inventario, params=params)

# Verificamos si la respuesta fue exitosa (código 200 significa que la solicitud fue correcta)
if response.status_code == 200:
    # Convertimos la respuesta en formato JSON para poder trabajar con los datos
    data_inventario = response.json()
    
    # Extraemos el enlace ('datos') que contiene la información detallada del inventario de estaciones
    link_inventario = data_inventario['datos']
    
    # Realizamos una segunda solicitud GET para acceder a los datos detallados en el enlace extraído
    inventario_response = requests.get(link_inventario)
    
    # Comprobamos si la solicitud del enlace fue exitosa
    if inventario_response.status_code == 200:
        # Guardamos los datos del inventario en una lista de estaciones en formato JSON
        estaciones = inventario_response.json()
        
        # Buscamos el código identificador ("indicativo") de la estación "MADRID, CIUDAD UNIVERSITARIA"
        indicativo = next((est['indicativo'] for est in estaciones if est['nombre'] == 'MADRID, CIUDAD UNIVERSITARIA'), None)
        
        # Imprimimos el indicativo de la estación para confirmar que lo encontramos
        print(f"Indicativo de la estación: {indicativo}")
        
        # 2. Obtener datos de climatología diaria
        # Si encontramos el indicativo, pasamos a solicitar los datos de climatología diaria de esa estación
        if indicativo:
            # Construimos la URL para la climatología diaria de la estación, especificando el rango de fechas y el indicativo
            url_climatologia_diaria = f'https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/2019-10-01T00:00:00UTC/fechafin/2019-10-30T23:59:59UTC/estacion/{indicativo}'
            
            # Pedimos a la API los datos de climatología diaria con la URL creada y los parámetros
            response_climatologia = requests.get(url_climatologia_diaria, params=params)
            
            # Comprobamos si la solicitud fue exitosa
            if response_climatologia.status_code == 200:
                # Convertimos la respuesta en JSON y extraemos el enlace a los datos detallados de climatología diaria
                data_climatologia = response_climatologia.json()
                link_climatologia = data_climatologia['datos']
                
                # Imprimimos el enlace a los datos de climatología diaria para tener una referencia
                # print("Link al fichero de datos de climatología diaria:", link_climatologia)

                # Pedimos los datos de climatología diaria usando el enlace que obtuvimos
                climatologia_response = requests.get(link_climatologia)
                
                # Verificamos si la solicitud de climatología diaria fue exitosa
                if climatologia_response.status_code == 200:
                    # Convertimos la respuesta a formato JSON y guardamos los datos de climatología
                    datos_climatologia = climatologia_response.json()
                    
                    # Imprimimos los datos de climatología diaria

                    print("\n---------------------------------------------------------------------------------------------------------------------------------\n")
                    print("Datos de climatología diaria:\n")
                    print("\n---------------------------------------------------------------------------------------------------------------------------------\n")
                    print(datos_climatologia)
                    
                    # Guardamos los datos en un archivo de texto para su posterior análisis o entrega
                    with open("climatologia_ciudad_universitaria_octubre_2019.txt", "w") as file:
                        file.write(str(datos_climatologia))
else:
    # Si la primera solicitud falla, imprimimos un mensaje de error con el código de respuesta
    print(f"Error: {response.status_code}, no se pudo obtener el inventario de estaciones")

Error: 401, no se pudo obtener el inventario de estaciones


----
## EJERCICIO 2

In [62]:
import requests  # Importamos la librería requests para realizar solicitudes a la API

# Creamos una lista vacía donde vamos a almacenar las temperaturas medias de cada año
temp_medias = []

# Definimos los valores de fecha para el inicio y fin de cada quincena de agosto
fecha_ini_1 = "-08-01T00:00:00UTC"  # La fecha de inicio será el 1 de agosto a las 00:00 UTC
fecha_fin_1 = "-08-15T23:59:59UTC"  # La fecha de fin será el 15 de agosto a las 23:59 UTC

# Nuestra clave API proporcionada por AEMET para autenticarnos en la API
api_key = ''

# Creamos un diccionario de parámetros que incluye nuestra clave API para autenticar cada petición
params = {'api_key': api_key}

# Empezamos un bucle para cada año del 2010 al 2019
for year in range(2010, 2020):        
    # Creamos las fechas de inicio y fin para el año en cuestión, concatenando el año con los valores de mes y día
    fecha_ini = str(year) + fecha_ini_1
    fecha_fin = str(year) + fecha_fin_1
    
    # Construimos la URL de la API para obtener la temperatura media de todas las estaciones para el año y fechas indicadas
    url = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_ini}/fechafin/{fecha_fin}/todasestaciones"
 
    # Realizamos la solicitud a la API
    response = requests.get(url, params=params)
    
    # Verificamos si la solicitud fue exitosa (código 200 significa que fue correcta)
    if response.status_code == 200:    
        # Obtenemos el enlace a los datos reales de la API desde la clave "datos" en la respuesta
        url_datos = response.json().get("datos")    
        
        # Realizamos una segunda solicitud para obtener los datos climáticos de todas las estaciones
        datos = requests.get(url_datos).json()     
 
        # Inicializamos variables para calcular la temperatura media de todas las estaciones para el año
        temps = 0  # Esta variable acumulará la suma de todas las temperaturas
        num_estaciones = 0   # Esta variable contará cuántas estaciones tienen datos de temperatura

        # Iniciamos un bucle sobre cada estación en los datos obtenidos
        for estacion in datos:
            # Obtenemos la temperatura media ("tmed") de la estación actual
            temp_estacion = estacion.get("tmed")
            
            # Verificamos que la temperatura no sea None (es decir, que sí haya un valor de temperatura)
            if temp_estacion != None:                                     
                # Reemplazamos la coma por un punto para poder convertir el valor a número decimal
                temp_estacion = float(temp_estacion.replace(",", "."))    
                
                # Aumentamos el contador de estaciones válidas
                num_estaciones += 1
                
                # Sumamos la temperatura de esta estación a la variable de acumulación
                temps += temp_estacion
 
        # Calculamos la temperatura media dividiendo la suma de temperaturas entre el número de estaciones con datos
        media = temps / num_estaciones
        
        # Añadimos la temperatura media del año actual al array de temperaturas medias
        temp_medias.append(media)
 

# Imprimimos un título para los resultados
print("\n---------------------------------------------------------------------------------------------------------------------------------\n")
print("\n TEMPERATURA MEDIA DE TODAS LAS ESTACIONES DE ESPAÑA EN LA PRIMERA QUINCENA DE LOS MESES DE AGOSTO COMPRENDIDOS ENTRE 2010 Y 2019\n")
print("\n---------------------------------------------------------------------------------------------------------------------------------\n")

# Recorremos el array de temperaturas medias y mostramos el año correspondiente y la media de temperatura
for i in range(len(temp_medias)):
    print(f"Año {2010 + i} Temperatura media: {temp_medias[i]:.4f} ºC")

# Imprimimos un título para mostrar el array completo
print("\n---------------------------------------------------------------------------------------------------------------------------------\n")
print("\nARRAY DE TEMPERATURAS\n")
print("\n---------------------------------------------------------------------------------------------------------------------------------\n")

# Mostramos el array de temperaturas medias completo
print(temp_medias)


---------------------------------------------------------------------------------------------------------------------------------


 TEMPERATURA MEDIA DE TODAS LAS ESTACIONES DE ESPAÑA EN LA PRIMERA QUINCENA DE LOS MESES DE AGOSTO COMPRENDIDOS ENTRE 2010 Y 2019


---------------------------------------------------------------------------------------------------------------------------------

Año 2010 Temperatura media: 23.2431 ºC
Año 2011 Temperatura media: 23.3516 ºC
Año 2012 Temperatura media: 23.9850 ºC
Año 2013 Temperatura media: 23.6762 ºC
Año 2014 Temperatura media: 22.4595 ºC
Año 2015 Temperatura media: 23.3967 ºC
Año 2016 Temperatura media: 23.6890 ºC
Año 2017 Temperatura media: 23.3032 ºC
Año 2018 Temperatura media: 24.7699 ºC
Año 2019 Temperatura media: 23.4475 ºC

---------------------------------------------------------------------------------------------------------------------------------


ARRAY DE TEMPERATURAS


---------------------------------------------------------